In [1]:
# !pip install -r requirements.txt

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import cv2
import json
import numpy as np
from PIL import Image
import os
import glob

# Prepare data

In [ ]:
image_paths = glob.glob('./data/*.jpg')
json_paths = [pth.replace('jpg', 'json') for pth in image_paths]
image_paths, json_paths

In [ ]:
image = cv2.imread(image_paths[1])
image_RGB = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_crop = image_RGB[540:, :, :]

plt.figure(figsize=(10,5))

plt.subplot(1, 2, 1)
plt.imshow(image_RGB)
plt.title("Image Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(image_crop)
plt.title("Image ")
plt.axis("off")

plt.show()

# Get Bbox

### Detect bbox by threshold

In [ ]:
# from segment import SegmentModel
from src.detect import DetectModel

model_detect = DetectModel(model_name='none')
# model_detect.predict(image_paths[0])

In [ ]:
path = image_paths[2]
masks = model_detect.predict(path)
image = cv2.imread(path)
H, W, _ = image.shape

for i, mask in enumerate(masks):
    x, y, w, h = mask
    # print(x,y,w,h)
    img = image[max(0, y-50):min(H,y+h+50), max(0,x-50):min(W,x+w+50)]
    cv2.imwrite(f"runs/detect/predict/output_{i}.jpg", img)
    # cv2.rectangle(image, (x-50, y-50), (x + w +50, y + h+50), (0, 255, 0), 2)

### Detect chili by model YOLO

- chưa load được model gọn nhẹ

# Segment for each chili

In [ ]:
from src.segment import SegmentModel

model_seg = SegmentModel(weight_path='./pretrain/yolov8_segment_v1.pt', model_name='yolo')

In [ ]:
import glob

img_paths = glob.glob('./runs/detect/predict/*.jpg')
imgs = [cv2.imread(path) for path in img_paths]
RGB_imgs = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in imgs]

results = model_seg.predict(RGB_imgs[0])

r = results['raw_results'][0]  # first image result

In [ ]:
[i['class_name'] for i in results['polygons']]

In [ ]:
names = r.names
masks = r.masks.data
polygons = r.masks.xy
xyxy = r.boxes.xyxy      # (N,4)
cls  = r.boxes.cls
conf = r.boxes.conf

In [ ]:
masks.shape

# Test pipeline

In [ ]:
from src.object import Fruits

fruits = Fruits(image_paths[0], model_detect, model_seg)
fruits.show()
# fruits.items[0].rseg

In [ ]:
fruits.items[0].rseg['raw_results'][0].masks.xy[0]
fruits.items[0].polygons[0].keys()

In [6]:
%cd Applying-Machine-Learning-for-Chili-Pepper-Phenotyping-and-Feature-Extraction-v2/

d:\Workspace\Project\chili_segment\Applying-Machine-Learning-for-Chili-Pepper-Phenotyping-and-Feature-Extraction-v2


In [ ]:
!python predict.py --input_path ./data/ --name_model_seg yolo --weight_seg ./pretrain/yolov8_segment_v1.pt --save_csv

python: can't open file "d:\\Workspace\\Project\\chili_segment\\Applying-Machine-Learning-for-Chili-Pepper-Phenotyping-and-Feature-Extraction-v2\\'D:\\Workspace\\Project\\chili_segment\\Applying-Machine-Learning-for-Chili-Pepper-Phenotyping-and-Feature-Extraction-v2\\predictor.py'": [Errno 22] Invalid argument


# Accuracy

# App demo